In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib
from scipy.constants import hbar
from scipy.integrate import dblquad

matplotlib.rcParams['animation.embed_limit'] = 2*1024*1024  # Zwiększenie limitu rozmiaru animacji


In [ ]:
# Funkcja falowa 2D z zależnością od czasu
def psi_2d_time(n_x, n_y, x, y, t, L):
    czesc_stacjonarna = (2 / L) * np.sin(n_x * np.pi * x / L) * np.sin(n_y * np.pi * y / L)
    E = (n_x**2 + n_y**2) * (np.pi**2) / (2 * L**2)
    czesc_czasowa = np.exp(-1j * E * t)

    return czesc_stacjonarna * czesc_czasowa

In [ ]:
# funkcja falowa w 2D z zależnością czasową i wykres w t=0
L = 1.0  # Wymiary studni
N = 100  # podział siatki
xs = np.linspace(0, L, N)
ys = np.linspace(0, L, N)
X, Y = np.meshgrid(xs, ys)


Z0 = psi_2d_time(2, 2, X, Y, 0, L)
plt.figure(figsize=(6, 5))

plt.pcolormesh(X, Y, Z0.real, shading='auto', cmap='viridis')

plt.colorbar(label='Amplituda') 
plt.title("Funkcja falowa 2D (Widok z góry)")
plt.show()



In [ ]:
# animacja czasowa funkcji falowej
n_x, n_y = 2, 2  

# Generowanie siatki
xs = np.linspace(0, L, N)
ys = np.linspace(0, L, N)
X, Y = np.meshgrid(xs, ys)


fig, ax = plt.subplots(figsize=(6, 6)) 


Z0 = psi_2d_time(n_x, n_y, X, Y, 0, L).real


mesh = ax.pcolormesh(X, Y, Z0, cmap='bwr', vmin=-2, vmax=2, shading='auto')
fig.colorbar(mesh, ax=ax, label='Amplituda $\\psi(x,y,t)$')


ax.set_xlim(0, L)
ax.set_ylim(0, L) 
ax.set_aspect('equal') 

dt = 0.01

def frame_psi(i):
    t = i * dt

    Z_complex = psi_2d_time(n_x, n_y, X, Y, t, L)
    Z_real = Z_complex.real
    
    
    mesh.set_array(Z_real.ravel())
    
    return mesh,


anim = FuncAnimation(fig, frame_psi, frames=200, interval=50, blit=True)


plt.close() 
HTML(anim.to_jshtml())

In [ ]:
#animacja gestosci prawdopodobienstwa
n_x, n_y = 2, 2
xs = np.linspace(0, L, N)
ys = np.linspace(0, L, N)
X, Y = np.meshgrid(xs, ys)


Z0 = psi_2d_time(n_x, n_y, X, Y, 0, L)
Z0_density = np.abs(Z0)**2

fig, ax = plt.subplots(figsize=(6, 5))


mesh = ax.pcolormesh(X, Y, Z0_density, cmap='inferno', vmin=0, vmax=np.max(Z0_density), shading='auto')
fig.colorbar(mesh, ax=ax, label='Gestosc Prawdopodobienstwa |Psi|^2')

ax.set_title(f"Stan stacjonarny ({n_x}, {n_y})")
ax.set_aspect('equal')

dt = 0.01

def frame_density(i):
    t = i * dt
    Psi = psi_2d_time(n_x, n_y, X, Y, t, L)
    
    Density = np.abs(Psi)**2
    
    mesh.set_array(Density.ravel())
    return mesh,


anim = FuncAnimation(fig, frame_density, frames=200, interval=50, blit=True)
plt.close()
HTML(anim.to_jshtml())


In [ ]:
# kombinacja liniowa stanów
def lincomb(X, Y, t, coeffs, L):
    norm = np.sqrt(sum(abs(c)**2 for c in coeffs.values()))
    psi_sum = np.zeros_like(X, dtype=complex)
    for (n_x, n_y), c in coeffs.items():
        psi_sum += c * psi_2d_time(n_x, n_y, X, Y, t, L)
    return psi_sum / norm


In [ ]:
# animacja czasowa kombinacji liniowej stanów
my_coeffs = {
    (1, 1): 0.8,
    (2, 2): 0.6
}

xs = np.linspace(0, L, N)
ys = np.linspace(0, L, N)
X, Y = np.meshgrid(xs, ys)

Z0 = lincomb(X, Y, 0, my_coeffs, L).real


fig, ax = plt.subplots(figsize=(6, 6)) 


Z0 = psi_2d_time(n_x, n_y, X, Y, 0, L).real


mesh = ax.pcolormesh(X, Y, Z0, cmap='bwr', vmin=-2, vmax=2, shading='auto')
fig.colorbar(mesh, ax=ax, label='Amplituda $\\psi(x,y,t)$')


ax.set_xlim(0, L)
ax.set_ylim(0, L) 
ax.set_aspect('equal')
plt.title(f"Kombinacja liniowa stanów {list(my_coeffs.keys())} ")



dt = 0.01

def frame_lincomb(i):
    t = i * dt
    
    Z = lincomb(X, Y, t, my_coeffs, L).real
    
    
    mesh.set_array(Z.ravel())
    return mesh,

anim = FuncAnimation(fig, frame_lincomb, frames=200, interval=50, blit=True)
plt.close() 
HTML(anim.to_jshtml())

In [ ]:
# animacja gestosci prawdopodobienstwa kombinacji liniowej stanów
coeffs = {
    (1, 3): 1.0,      
    (2, 2): 1.5        
}


fig, ax = plt.subplots(figsize=(8, 5))


Z0 = lincomb(X, Y, 0, coeffs, L)
Z0_density = np.abs(Z0)**2


mesh = ax.pcolormesh(X, Y, Z0_density, cmap='inferno', 
                     vmin=0, vmax=np.max(Z0_density), shading='auto')
fig.colorbar(mesh, ax=ax, label=r'Gęstość $|\Psi_{total}|^2$')
ax.set_title(f"Kombinacja liniowa stanów {list(coeffs.keys())} - Gęstość prawdopodobieństwa")
ax.set_aspect('equal')

dt = 0.02

def frame_lin_density(i):
    t = i * dt 
    
    
    Psi_total = lincomb(X, Y, t, coeffs, L)
    Density = np.abs(Psi_total)**2
    
    mesh.set_array(Density.ravel())
    return mesh,

anim = FuncAnimation(fig, frame_lin_density, frames=200, interval=30, blit=True)
plt.close()
HTML(anim.to_jshtml())



In [ ]:
dt = 0.0002

dx = xs[1] - xs[0]


x0, y0 = 0.3, 0.3   
sigma = 0.12            
kx, ky = 8, 8    



def calka_rzeczywista(y, x, nx, ny):
    psi_n = (2/L) * np.sin(nx * np.pi * x / L) * np.sin(ny * np.pi * y / L)
    psi_cel = np.exp(-((x - x0)**2 + (y - y0)**2) / (2 * sigma**2)) * np.exp(1j * (kx * x + ky * y))
    return np.real(np.conj(psi_n) * psi_cel)

def calka_urojona(y, x, nx, ny):
    psi_n = (2/L) * np.sin(nx * np.pi * x / L) * np.sin(ny * np.pi * y / L)
    psi_cel = np.exp(-((x - x0)**2 + (y - y0)**2) / (2 * sigma**2)) * np.exp(1j * (kx * x + ky * y))
    return np.imag(np.conj(psi_n) * psi_cel)

N_max = 25 
coeffs_gauss = {} 

for nx in range(1, N_max + 1):
    for ny in range(1, N_max + 1):
        real, _ = dblquad(calka_rzeczywista, 0, L, 0, L, args=(nx, ny))
        imag, _ = dblquad(calka_urojona, 0, L, 0, L, args=(nx, ny))
        overlap = real + 1j * imag
        
        if np.abs(overlap) > 0.00005:
            coeffs_gauss[(nx, ny)] = overlap

print("Obliczono współczynniki dla", len(coeffs_gauss), "stanów własnych.")
fig, ax = plt.subplots(figsize=(6, 6))


Z0 = lincomb(X, Y, 0, coeffs_gauss, L).real

vmax_amplituda = np.max(np.abs(Z0)) * 1.2

mesh = ax.pcolormesh(X, Y, Z0, cmap='viridis', vmin= -vmax_amplituda, vmax=vmax_amplituda, shading='auto')
fig.colorbar(mesh, ax=ax, label='Amplituda $\\psi(x,y,t)$')

ax.set_title("Paczka Falowa ")
ax.set_xlim(0, L); ax.set_ylim(0, L)
ax.set_aspect('equal')

def frame_pack(i):
    t = i * dt
    
    pack_time = lincomb(X, Y, t, coeffs_gauss, L).real
    

    mesh.set_array(pack_time.ravel())
    return mesh,

anim = FuncAnimation(fig, frame_pack, frames=300, interval=30, blit=True)
plt.close()
HTML(anim.to_jshtml())

In [ ]:
N = 100
xs = np.linspace(0, L, N)
ys = np.linspace(0, L, N)
X, Y = np.meshgrid(xs, ys)
dt = 0.0005

dx = xs[1] - xs[0]


x0, y0 = 0.3, 0.3   
sigma = 0.12            
kx, ky = 4, 6    

N_max = 20 
coeffs_gauss = {} 

for nx in range(1, N_max + 1):
    for ny in range(1, N_max + 1):
        real, _ = dblquad(calka_rzeczywista, 0, L, 0, L, args=(nx, ny))
        imag, _ = dblquad(calka_urojona, 0, L, 0, L, args=(nx, ny))
        overlap = real + 1j * imag
        
        if np.abs(overlap) > 0.00005:
            coeffs_gauss[(nx, ny)] = overlap

print("Obliczono współczynniki dla", len(coeffs_gauss), "stanów własnych.")
fig, ax = plt.subplots(figsize=(6, 6), dpi=150)


Z0 = np.abs(lincomb(X, Y, 0, coeffs_gauss, L))**2

vmax_density = np.max(Z0) * 1.2

mesh = ax.pcolormesh(X, Y, Z0, cmap='inferno', vmin=0, vmax=1, shading='gouraud')
fig.colorbar(mesh, ax=ax, label='Gęstość prawdopodobieństwa')

ax.set_title("Paczka Falowa ")
ax.set_xlim(0, L); ax.set_ylim(0, L)
ax.set_aspect('equal')

def frame_pack_density(i):
    t = i * dt
    
    pack_time = lincomb(X, Y, t, coeffs_gauss, L)
    Density = np.abs(pack_time)**2

    mesh.set_array(Density.ravel())
    return mesh,

anim = FuncAnimation(fig, frame_pack_density, frames=400, interval=20, blit=True)
plt.close()
HTML(anim.to_jshtml())